# Juan Diego Cadena, 329220

## 05 - Selección y Model Registry

El holdout de abril-septiembre de 2026 se abre una sola vez para el ganador y el segundo. El umbral intenta reducir por lo menos 10% los picos no anticipados frente a la regla estacional, manteniendo una precisión mínima; como respaldo minimiza `15*FN + FP`.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Image, display
from src.experimentos import *
df, data, info = cargar_contexto()
print(f'Fuente: {info.source}')
print(f'OBT: {info.rows:,} filas x {info.columns} columnas; corte {info.snapshot_end}')
print(f'Train: {len(data.train):,}; holdout: {len(data.test):,}; target positivo: {data.train[data.target_column].mean():.2%}')
print(f'Validación: 4 semestres completos; purga: {data.purge_days} días; features: {len(data.feature_columns)}')


Fuente: C:\Universidad\Data_Science\Data_science_end_to_end\laboratorios\semana-09-sesion-01\data\obt_emergencias_canton_dia.csv
OBT: 429,632 filas x 37 columnas; corte 2026-09-30
Train: 387,968; holdout: 40,320; target positivo: 8.17%
Validación: 4 semestres completos; purga: 3 días; features: 48


In [2]:
seleccion = ejecutar_seleccion(data, info, df, force=False)
seleccion

{'champion': 'hist_gradient_boosting',
 'challenger': 'catboost',
 'champion_run_id': 'b94b74256a114988b926217055c66134',
 'challenger_run_id': '07a10eaa99a74a6dafc8b8b4800b0f26',
 'champion_version': 3,
 'challenger_version': 4,
 'model_registry_name': 'ecu911-alta-demanda-modelo',
 'champion_threshold': 0.35,
 'champion_test_pr_auc': 0.26925052381184034,
 'champion_test_accuracy': 0.8274305555555556,
 'champion_cv_pr_auc_ponderado': 0.2315549938988667,
 'champion_reproduced_pr_auc': 0.26925052381184034,
 'business_missed_peak_reduction': 0.3626501356063542,
 'seasonal_baseline_test': {'accuracy': 0.8422123015873015,
  'pr_auc': 0.12085130152491766,
  'precision': 0.13813540004558925,
  'recall': 0.19014747411358646,
  'false_positives': 3781,
  'false_negatives': 2581,
  'true_positives': 606,
  'true_negatives': 33352,
  'weighted_error_cost': 42496.0,
  'alert_rate': 0.1088045634920635},
 'validation_design': {'folds': 4,
  'block': 'semestres_completos',
  'purge_days': 3,
  'seas

In [3]:
import mlflow, mlflow.sklearn
from sklearn.metrics import average_precision_score
configurar_mlflow()
modelo=mlflow.sklearn.load_model(f"models:/{seleccion['model_registry_name']}@champion")
score=modelo.predict_proba(data.test[data.feature_columns])[:,1]
reproducida=average_precision_score(data.test[data.target_column],score)
assert abs(reproducida-seleccion['champion_test_pr_auc'])<1e-12
print(f"PR-AUC reproducida desde @champion: {reproducida:.6f}")

PR-AUC reproducida desde @champion: 0.269251


In [4]:
import pandas as pd
pd.read_csv(ROOT/'docs'/'finalistas_test.csv')

,modelo,rol,run_id,threshold,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_ponderado_mean,cv_pr_auc_feriados_mean,cv_pr_auc_fin_anio_mean,test_pr_auc,test_pr_auc_ponderado,test_accuracy,test_precision,test_recall,test_false_negatives,test_false_positives,business_missed_peak_reduction,train_fit_seconds,test_inference_seconds
0,hist_gradient_boosting,champion,b94b74256a114988b926217055c66134,0.35,0.183827,0.042729,0.231555,0.540549,0.324408,0.269251,0.267713,0.827431,0.224945,0.483841,1645,5313,0.362650,21.301659,0.575981
1,catboost,challenger,07a10eaa99a74a6dafc8b8b4800b0f26,0.52,0.178665,0.041661,0.227624,0.531455,0.314436,0.239372,0.235836,0.839881,0.232269,0.444933,1769,4687,0.314607,14.776694,0.571910


In [5]:
pd.read_csv(ROOT/'docs'/'pruebas_estacionales.csv').query("alcance == 'backtests_oof'")

,modelo,alcance,fold,segmento,filas,positivos,prevalencia,accuracy,pr_auc,precision,recall,false_positives,false_negatives,true_positives,true_negatives,weighted_error_cost,alert_rate
4,hist_gradient_boosting,backtests_oof,NaN,todos,163744,9691,0.059184,0.902366,0.184487,0.201894,0.219998,8428,7559,2132,145625,121813.0,0.064491
5,hist_gradient_boosting,backtests_oof,NaN,feriado,6496,1131,0.174107,0.719674,0.549883,0.351804,0.724138,1509,312,819,3856,6189.0,0.358374
6,hist_gradient_boosting,backtests_oof,NaN,ventana_feriado,22400,2297,0.102545,0.822589,0.380456,0.288097,0.496300,2817,1157,1140,17286,20172.0,0.176652
7,hist_gradient_boosting,backtests_oof,NaN,fin_anio,8512,960,0.112782,0.847392,0.528490,0.387674,0.609375,924,375,585,6628,6549.0,0.177279
8,hist_gradient_boosting,backtests_oof,NaN,carnaval_semana_santa,1344,379,0.281994,0.606399,0.468212,0.403101,0.823219,462,67,312,503,1467.0,0.575893
9,hist_gradient_boosting,backtests_oof,NaN,diciembre,13888,1065,0.076685,0.881192,0.224432,0.265437,0.310798,916,734,331,11907,11926.0,0.089790
10,hist_gradient_boosting,backtests_oof,NaN,regular,141344,7394,0.052312,0.915009,0.114574,0.150235,0.134163,5611,6402,992,128339,101641.0,0.046716
15,catboost,backtests_oof,NaN,todos,163744,9691,0.059184,0.901798,0.178710,0.199398,0.218656,8508,7572,2119,145545,122088.0,0.064900
16,catboost,backtests_oof,NaN,feriado,6496,1131,0.174107,0.687962,0.533889,0.330431,0.771883,1769,258,873,3596,5639.0,0.406712
17,catboost,backtests_oof,NaN,ventana_feriado,22400,2297,0.102545,0.799420,0.373079,0.264681,0.537658,3431,1062,1235,16672,19361.0,0.208304


También se entrena una sensibilidad t+7 con la familia ganadora. La selección principal queda en t+3 porque es el horizonte operativo prioritario del PSet 1; t+7 se registra por separado para medir la degradación.